In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
%cd /content/drive/MyDrive/LLM-jax

/content/drive/MyDrive/LLM-jax


In [49]:
!ls


L2  L3	L4  L5	sample.py


In [50]:
!pip install "jax[cuda12]==0.6.2" flax==0.10.7 grain==0.2.13 tiktoken==0.7.0 gradio==6.5.1 orbax-checkpoint matplotlib -q

In [51]:
%cd /content/drive/MyDrive/LLM-jax/L3
!ls

/content/drive/MyDrive/LLM-jax/L3
helper.py  L3.ipynb  __pycache__  requirements.txt  TinyStories-1000.txt


In [52]:
import sys
sys.path.insert(0, '.')

from helper import load_stories_from_file, StoryDataset, load_and_preprocess_data
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
stories = load_stories_from_file("TinyStories-1000.txt", max_stories=100)
print(f"Loaded {len(stories)} stories")
print("First story preview:", stories[0][:100])

Loading stories from TinyStories-1000.txt...
Loaded 100 stories
Loaded 100 stories
First story preview: One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with


new cells


In [53]:
fixed_helper = '''
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import grain.python as pygrain
import optax
import tiktoken
from pathlib import Path

tokenizer = tiktoken.get_encoding("gpt2")
vocab_size = tokenizer.n_vocab
num_transformer_blocks = 6
maxlen = 128
embed_dim = 192
num_heads = 6
feed_forward_dim = int(2/3 * 4 * embed_dim)
batch_size = 32
num_epochs = 10

class TransformerBlock(nnx.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, *, rngs):
        self.attention = nnx.MultiHeadAttention(
            num_heads=num_heads, in_features=embed_dim,
            qkv_features=embed_dim, out_features=embed_dim,
            decode=False, rngs=rngs
        )
        self.ff1   = nnx.Linear(embed_dim, ff_dim, rngs=rngs)
        self.ff2   = nnx.Linear(ff_dim, embed_dim, rngs=rngs)
        self.norm1 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.norm2 = nnx.LayerNorm(embed_dim, rngs=rngs)

    def __call__(self, x, mask=None):
        attn_out = self.attention(x, mask=mask)
        x = self.norm1(x + attn_out)
        ff_out = self.ff2(nnx.gelu(self.ff1(x)))
        x = self.norm2(x + ff_out)
        return x

class TokenAndPositionEmbedding(nnx.Module):
    def __init__(self, maxlen, vocab_size, embed_dim, *, rngs):
        self.token_emb = nnx.Embed(vocab_size, embed_dim, rngs=rngs)
        self.pos_emb   = nnx.Embed(maxlen, embed_dim, rngs=rngs)

    def __call__(self, x):
        seq_len = x.shape[1]
        positions = jnp.arange(seq_len)[None, :]
        return self.token_emb(x) + self.pos_emb(positions)

class MiniGPT(nnx.Module):
    def __init__(self, maxlen=maxlen, vocab_size=vocab_size, embed_dim=embed_dim,
                 num_heads=num_heads, feed_forward_dim=feed_forward_dim,
                 num_transformer_blocks=num_transformer_blocks, *, rngs=nnx.Rngs(0)):
        self.maxlen    = maxlen
        self.embedding = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim, rngs=rngs)
        self.transformer_blocks = [
            TransformerBlock(embed_dim, num_heads, feed_forward_dim, rngs=rngs)
            for _ in range(num_transformer_blocks)
        ]
        self.output_layer = nnx.Linear(embed_dim, vocab_size, use_bias=False, rngs=rngs)

    def causal_attention_mask(self, seq_len):
        return jnp.tril(jnp.ones((seq_len, seq_len)))

    def __call__(self, token_ids):
        seq_len = token_ids.shape[1]
        mask    = self.causal_attention_mask(seq_len)
        x       = self.embedding(token_ids)
        for block in self.transformer_blocks:
            x = block(x, mask=mask)
        return self.output_layer(x)

def generate_text(model, start_tokens, max_new_tokens=50, temperature=1.0):
    tokens = list(start_tokens)
    for _ in range(max_new_tokens):
        context    = tokens[-model.maxlen:]
        actual_len = len(context)
        if actual_len < model.maxlen:
            context = context + [0] * (model.maxlen - actual_len)
        logits     = model(jnp.array(context)[None, :])
        next_logits = logits[0, actual_len - 1, :] / temperature
        next_token  = int(jnp.argmax(next_logits))
        if next_token == tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]:
            break
        tokens.append(next_token)
    return tokenizer.decode(tokens)

def generate_story(model, story_prompt, temperature, max_new_tokens):
    start_tokens = tokenizer.encode(story_prompt)[:maxlen]
    return generate_text(model, start_tokens, max_new_tokens=max_new_tokens, temperature=temperature)

class StoryDataset:
    def __init__(self, stories, maxlen, tokenizer):
        self.stories   = stories
        self.maxlen    = maxlen
        self.tokenizer = tokenizer
        self.end_token = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]
    def __len__(self): return len(self.stories)
    def __getitem__(self, idx):
        tokens = self.tokenizer.encode(self.stories[idx], allowed_special={"<|endoftext|>"})
        if len(tokens) > self.maxlen: tokens = tokens[:self.maxlen]
        tokens.extend([0] * (self.maxlen - len(tokens)))
        return tokens

def load_stories_from_file(file_path, max_stories=None):
    file_path = Path(file_path)
    stories, current = [], []
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if "<|endoftext|>" in line:
                parts = line.split("<|endoftext|>")
                for part in parts[:-1]:
                    current.append(part)
                    text = "".join(current).strip()
                    if text: stories.append(text + "<|endoftext|>")
                    if max_stories and len(stories) >= max_stories: return stories
                    current = []
                if parts[-1].strip(): current = [parts[-1]]
            else:
                current.append(line)
    if current:
        text = "".join(current).strip()
        if text: stories.append(text + "<|endoftext|>")
    return stories

def load_and_preprocess_data(file_path, batch_size, maxlen, max_stories=100_000,
                              num_epochs=1, shuffle=False, seed=42):
    stories = load_stories_from_file(file_path, max_stories)
    print(f"Loaded {len(stories)} stories")
    dataset = StoryDataset(stories, maxlen, tokenizer)
    estimated_batches = len(dataset) // batch_size
    sampler = pygrain.IndexSampler(num_records=len(dataset), shuffle=shuffle,
                                   seed=seed, shard_options=pygrain.NoSharding(),
                                   num_epochs=num_epochs)
    dataloader = pygrain.DataLoader(data_source=dataset, sampler=sampler,
                                    operations=[pygrain.Batch(batch_size=batch_size, drop_remainder=True)])
    return dataloader, estimated_batches
'''

with open("/content/drive/MyDrive/LLM-jax/L4/helper.py", "w") as f:
    f.write(fixed_helper)
with open("/content/drive/MyDrive/LLM-jax/L5/helper.py", "w") as f:
    f.write(fixed_helper)

print("helper.py fixed and saved to L4 and L5!")

helper.py fixed and saved to L4 and L5!


In [54]:
import sys, jax, jax.numpy as jnp, optax
import flax.nnx as nnx

sys.path.insert(0, '/content/drive/MyDrive/LLM-jax/L4')
from helper import MiniGPT, load_and_preprocess_data, generate_text

text_dl, batches_per_epoch = load_and_preprocess_data(
    file_path='/content/drive/MyDrive/LLM-jax/L4/TinyStories-1000.txt',
    batch_size=32, maxlen=128, max_stories=1000, shuffle=True, seed=42
)
print(f"Batches per epoch: {batches_per_epoch}")

Loading data from /content/drive/MyDrive/LLM-jax/L4/TinyStories-1000.txt (max 1,000 stories)
Loaded 1,000 stories
Estimated batches per epoch: 31
Created DataLoader with batch_size=32, maxlen=128
Batches per epoch: 31


In [55]:
%cd /content/drive/MyDrive/LLM-jax/L4

/content/drive/MyDrive/LLM-jax/L4


In [56]:
# Run this cell BEFORE building the model — it patches the TransformerBlock
import jax.numpy as jnp
import flax.nnx as nnx

class TransformerBlock(nnx.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, *, rngs):
        self.attention = nnx.MultiHeadAttention(
            num_heads=num_heads,
            in_features=embed_dim,
            qkv_features=embed_dim,
            out_features=embed_dim,
            decode=False,
            rngs=rngs
        )
        # These were MISSING — this is why loss was stuck at 5.4
        self.ff1 = nnx.Linear(embed_dim, ff_dim, rngs=rngs)
        self.ff2 = nnx.Linear(ff_dim, embed_dim, rngs=rngs)
        self.norm1 = nnx.LayerNorm(embed_dim, rngs=rngs)
        self.norm2 = nnx.LayerNorm(embed_dim, rngs=rngs)

    def __call__(self, x, mask=None):
        # Attention + residual + norm
        attn_out = self.attention(x, mask=mask)
        x = self.norm1(x + attn_out)
        # FeedForward + residual + norm
        ff_out = self.ff2(nnx.gelu(self.ff1(x)))
        x = self.norm2(x + ff_out)
        return x

# Override the broken class from helper.py
import helper
helper.TransformerBlock = TransformerBlock
print("TransformerBlock fixed!")

TransformerBlock fixed!


In [57]:
model = MiniGPT()

def loss_fn(model, batch):
    inputs, targets = batch
    logits = model(inputs)
    return optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean(), logits

num_epochs = 15
total_steps  = batches_per_epoch * num_epochs
warmup_steps = max(1, total_steps // 10)

lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0, peak_value=5e-4,
    warmup_steps=warmup_steps, decay_steps=total_steps, end_value=1e-5
)
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=lr_schedule, weight_decay=0.01))
metrics   = nnx.MultiMetric(loss=nnx.metrics.Average('loss'))

@nnx.jit
def train_step(model, optimizer, metrics, batch):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch)
    metrics.update(loss=loss, logits=logits, labels=batch[1])
    optimizer.update(grads)

metrics_history = {'train_loss': []}
prep_target = jax.vmap(lambda t: jnp.concatenate((t[1:], jnp.array([0]))))

for epoch in range(num_epochs):
    for step, batch in enumerate(text_dl):
        ib = jnp.array(jnp.array(batch).T).astype(jnp.int32)
        tb = prep_target(jnp.array(jnp.array(batch).T)).astype(jnp.int32)
        train_step(model, optimizer, metrics, (ib, tb))
        if (step + 1) % 2 == 0:
            for k, v in metrics.compute().items():
                metrics_history[f'train_{k}'].append(v)
            metrics.reset()
    print(f"Epoch {epoch+1}/{num_epochs} — Loss: {metrics_history['train_loss'][-1]:.4f}")

print("Done!")

Epoch 1/15 — Loss: 8.9954
Epoch 2/15 — Loss: 6.0547
Epoch 3/15 — Loss: 5.0962
Epoch 4/15 — Loss: 4.5858
Epoch 5/15 — Loss: 4.2263
Epoch 6/15 — Loss: 3.9652
Epoch 7/15 — Loss: 3.7382
Epoch 8/15 — Loss: 3.5743
Epoch 9/15 — Loss: 3.4339
Epoch 10/15 — Loss: 3.3114
Epoch 11/15 — Loss: 3.2334
Epoch 12/15 — Loss: 3.1599
Epoch 13/15 — Loss: 3.1204
Epoch 14/15 — Loss: 3.0891
Epoch 15/15 — Loss: 3.0768
Done!


In [58]:
from pathlib import Path
import orbax

checkpoint_path = Path.cwd() / "model_checkpoint.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()
checkpointer.save(checkpoint_path, nnx.state(model), force=True)
print(f"Saved to {checkpoint_path}")

Saved to /content/drive/MyDrive/LLM-jax/L4/model_checkpoint.orbax


In [59]:
%cd /content/drive/MyDrive/LLM-jax/L5

import jax
import flax.nnx as nnx
import orbax
from orbax import checkpoint
from jax.sharding import SingleDeviceSharding
from pathlib import Path
import sys
sys.path.insert(0, '.')

from helper import MiniGPT, generate_story

model = MiniGPT()

cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

# Point to where L4 saved the checkpoint
checkpoint_path = Path('/content/drive/MyDrive/LLM-jax/L4/model_checkpoint.orbax')
checkpointer = orbax.checkpoint.PyTreeCheckpointer()
restored_state = checkpointer.restore(checkpoint_path, item=nnx.state(model), restore_args=restore_args)
nnx.update(model, restored_state)
print("Model loaded!")

/content/drive/MyDrive/LLM-jax/L5
Model loaded!


In [60]:
result = generate_story(model, "Once upon a time a little girl", temperature=0.2, max_new_tokens=50)
print(result)

Once upon a time a little girl named Tim who loved to play with her friends. One day, they went to the park, she saw a big house with her mom. She wanted to the park, but it was a big, but it was very happy and said.
The


In [61]:
import gradio as gr

def create_story(story_prompt, temperature, max_new_tokens):
    return generate_story(model, story_prompt, temperature, int(max_new_tokens))

demo = gr.Interface(
    fn=create_story,
    inputs=[
        gr.Textbox(label="Story Prompt"),
        gr.Slider(minimum=0, maximum=1.0, value=0.8, step=0.01, label="Temperature"),
        gr.Slider(minimum=0, maximum=200, value=30, step=1, label="Max Tokens")
    ],
    outputs=["text"]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://81213986d11744491a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
